# OpenMeet visual model: gaze estimation by transfer learning

Part of the reading-gaze (answer-integrity) signal on `feature/gaze-detection`. This notebook trains an
**appearance-based gaze estimator**: a CNN that takes a normalised face crop and returns gaze pitch and yaw. Its
per-frame output replaces the MediaPipe iris offsets as the input of the existing reading detector
(`readgaze/`), which looks for the sawtooth pattern of reading.

**Why this model and not a "cheating classifier".** We have no labelled video of people reading answers yet
(GazeBase is eye-tracker coordinates; our own recordings are Phase 3). A video-level classifier can't be trained
honestly. The gaze estimator can: MPIIFaceGaze has 37,667 laptop-webcam face images of 15 people with gaze
labels, and gaze-pretrained backbones exist.

**Transfer learning.** Three starting points are compared, all fine-tuned end to end:
ImageNet ResNet-18 (timm), and ResNet-18 / ResNet-50 already trained for gaze on ETH-XGaze
(110 other people, so nothing leaks into MPIIFaceGaze).

**Protocol** (fixed before results):
- Split by person only. Development: 3 folds of 5 held-out people. Final number: leave-one-person-out (LOPO,
  15 folds), the standard MPIIFaceGaze protocol. Headline metric: mean over people of the mean angular error (deg).
- Robustness: test faces shrunk to 160–32 px wide and JPEG-compressed (q30), to mimic a small, compressed
  meeting tile.
- Baseline: the current OpenMeet pipeline (MediaPipe iris offsets + linear map) on the same faces.

**Run it:** Kaggle, *GPU T4 x2*, *Internet on*. Everything is downloaded from the original sources.
The cells run top to bottom; training jobs run in the background, one per GPU, and `JOBS.join()` waits for them.

**Data terms.** MPIIFaceGaze and ETH-XGaze models are for non-commercial research (CC BY-NC-SA 4.0). Cite
Zhang et al. 2017 (*It's written all over your face*) and Zhang et al. 2020 (*ETH-XGaze*). Don't re-upload the
data or the trained weights publicly.

## 1. Data
MPIIFaceGaze from the official MPI server, then the standard data normalisation.

In [ ]:
# Environment check and MPIIFaceGaze download (official MPI server, 942 MB)
import os, subprocess, time
import torch, timm, cv2, numpy as np
print('torch', torch.__version__, '| timm', timm.__version__, '| cv2', cv2.__version__, '| numpy', np.__version__)
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], os.cpu_count(), 'CPUs',
      round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30, 1), 'GB RAM')
os.makedirs('/tmp/mpii', exist_ok=True)
t = time.time()
if not os.path.isdir('/tmp/mpii/MPIIFaceGaze'):
    subprocess.run('cd /tmp/mpii && wget -q -c https://datasets.d2.mpi-inf.mpg.de/MPIIGaze/MPIIFaceGaze.zip && unzip -q -o MPIIFaceGaze.zip && rm MPIIFaceGaze.zip',
                   shell=True, check=True)
print('download + unzip:', round(time.time() - t), 's')
print(subprocess.run('wc -l /tmp/mpii/MPIIFaceGaze/p*/p*.txt | tail -1', shell=True, capture_output=True, text=True).stdout)
print(open('/tmp/mpii/MPIIFaceGaze/p00/p00.txt').readline())

In [ ]:
# data normalisation (Sugano 2014 / Zhang 2018): undistort, then warp each face into a virtual
# camera 600 mm away (224x224, focal 960). Gaze is rotated only. Same framing as ETH-XGaze models.
import numpy as np, cv2, scipy.io as sio, os, time
from concurrent.futures import ProcessPoolExecutor
ROOT, CACHE = '/tmp/mpii/MPIIFaceGaze', '/tmp/mpii_cache'
SIZE, FOCAL, DIST = 224, 960.0, 600.0
CAM_N = np.array([[FOCAL, 0, SIZE / 2], [0, FOCAL, SIZE / 2], [0, 0, 1.0]])
PEOPLE = [f'p{i:02d}' for i in range(15)]

def vec_to_pitchyaw(g):
    return np.array([np.arcsin(-g[1]), np.arctan2(-g[0], -g[2])])

def normalise(img, hr, fc, gt, cam):
    d = np.linalg.norm(fc)
    hR = cv2.Rodrigues(hr)[0]
    forward = fc / d
    down = np.cross(forward, hR[:, 0]); down /= np.linalg.norm(down)
    right = np.cross(down, forward); right /= np.linalg.norm(right)
    R = np.stack([right, down, forward])
    W = CAM_N @ np.diag([1.0, 1.0, DIST / d]) @ R @ np.linalg.inv(cam)
    face = cv2.warpPerspective(img, W, (SIZE, SIZE))
    g = R @ (gt - fc); g /= np.linalg.norm(g)
    h = (R @ hR)[:, 2]
    return face, vec_to_pitchyaw(g), np.array([np.arcsin(h[1]), np.arctan2(h[0], h[2])])

def process_person(p):
    calib = sio.loadmat(f'{ROOT}/{p}/Calibration/Camera.mat')
    cam, dist = calib['cameraMatrix'], calib['distCoeffs']
    lines = [l.split() for l in open(f'{ROOT}/{p}/{p}.txt').read().strip().splitlines()]
    faces = np.zeros((len(lines), SIZE, SIZE, 3), np.uint8)
    gaze, head = np.zeros((len(lines), 2), np.float32), np.zeros((len(lines), 2), np.float32)
    maps = {}
    for i, l in enumerate(lines):
        v = np.array(l[1:27], dtype=np.float64)
        img = cv2.imread(f'{ROOT}/{p}/{l[0]}')
        key = img.shape[:2]
        if key not in maps:
            maps[key] = cv2.initUndistortRectifyMap(cam, dist, None, cam, (key[1], key[0]), cv2.CV_32FC1)
        img = cv2.remap(img, *maps[key], cv2.INTER_LINEAR)
        f, g, h = normalise(img, v[14:17], v[20:23], v[23:26], cam)
        faces[i], gaze[i], head[i] = f[:, :, ::-1], g, h
    return p, faces, gaze, head, [l[0] for l in lines], [l[27] for l in lines]

def build_cache():
    os.makedirs(CACHE, exist_ok=True)
    t = time.time()
    counts = [sum(1 for _ in open(f'{ROOT}/{p}/{p}.txt')) for p in PEOPLE]
    out = np.lib.format.open_memmap(f'{CACHE}/faces.npy', mode='w+', dtype=np.uint8, shape=(sum(counts), SIZE, SIZE, 3))
    gaze, head, person, files, eye, off = [], [], [], [], [], 0
    with ProcessPoolExecutor(os.cpu_count()) as ex:
        for p, f, g, h, names, e in ex.map(process_person, PEOPLE):
            out[off:off + len(f)] = f; off += len(f)
            gaze.append(g); head.append(h); person += [int(p[1:])] * len(f); files += names; eye += e
            print(p, len(f), f'{time.time() - t:.0f}s', end=' | ', flush=True)
    out.flush(); del out
    np.savez(f'{CACHE}/meta.npz', gaze=np.concatenate(gaze), head=np.concatenate(head), person=np.array(person),
             files=np.array(files), eye=np.array(eye))  # written last: its presence means the cache is complete
    print('\ncached', off, 'faces in', round(time.time() - t), 's')

if os.path.exists(f'{CACHE}/meta.npz'):
    print('cache already built; not rebuilding (training jobs may be reading it):', CACHE)
else:
    build_cache()

## 2. Training library
`gazelib.py` (model builder, GPU augmentation, training loop, prediction) and `run.py` (one cross-validation fold per process).

In [ ]:
%%writefile /kaggle/working/gazelib.py
# OpenMeet visual model: appearance-based gaze estimation by transfer learning on MPIIFaceGaze.
import math, queue, threading, time
import cv2, numpy as np, timm, torch, torch.nn as nn, torch.nn.functional as F

CACHE = '/tmp/mpii_cache'
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
ETH = ('hysts/ptgaze-eth-xgaze-resnet18', 'model.safetensors')


def load():
    m = np.load(f'{CACHE}/meta.npz')
    return np.load(f'{CACHE}/faces.npy', mmap_mode='r'), m['gaze'].astype(np.float32), m['person']


def to_vec(py):
    p, y = py[:, 0], py[:, 1]
    return torch.stack([-torch.cos(p) * torch.sin(y), -torch.sin(p), -torch.cos(p) * torch.cos(y)], 1)


def ang_err(pred, true):
    # Angular error in degrees between (pitch, yaw) arrays in radians.
    a = to_vec(torch.as_tensor(np.asarray(pred), dtype=torch.float64))
    b = to_vec(torch.as_tensor(np.asarray(true), dtype=torch.float64))
    cos = (a * b).sum(1) / (a.norm(dim=1) * b.norm(dim=1))
    return torch.rad2deg(torch.arccos(cos.clamp(-1, 1))).numpy()


def build(cfg):
    name = cfg['backbone']
    if name == 'eth_resnet18':  # ResNet-18 already trained for gaze on ETH-XGaze (other people)
        from huggingface_hub import hf_hub_download
        from safetensors.torch import load_file
        model = timm.create_model('resnet18', pretrained=False, num_classes=2)
        sd = load_file(hf_hub_download(*ETH))
        for pre in ('model.', 'module.', 'backbone.'):
            sd = {k[len(pre):] if k.startswith(pre) else k: v for k, v in sd.items()}
        model.load_state_dict(sd)
        return model
    if name == 'eth_resnet50':  # official ETH-XGaze ResNet-50, converted to safetensors in the notebook
        from safetensors.torch import load_file
        model = timm.create_model('resnet50', pretrained=False, num_classes=2)
        model.load_state_dict(load_file('/tmp/eth_xgaze_r50.safetensors'))
        return model
    kw = {'drop_path_rate': cfg['drop_path']} if cfg.get('drop_path') else {}
    return timm.create_model(name, pretrained=cfg.get('pretrained', True), num_classes=2, **kw)


def batches(faces, labels, idx, bs, shuffle, rng=None):
    # Background-thread batch loader from the memory-mapped face array.
    order = rng.permutation(idx) if shuffle else np.asarray(idx)
    n = len(order) // bs * bs if shuffle else len(order)
    if torch.is_tensor(faces):  # dataset already on the GPU: index there, no CPU copies
        for i in range(0, n, bs):
            b = np.sort(order[i:i + bs])
            yield faces[torch.from_numpy(b).to(faces.device)], torch.from_numpy(labels[b]), b
        return
    q = queue.Queue(maxsize=6)

    def work():
        for i in range(0, n, bs):
            b = np.sort(order[i:i + bs])
            q.put((torch.from_numpy(np.ascontiguousarray(faces[b])).pin_memory(), torch.from_numpy(labels[b]), b))
        q.put(None)

    threading.Thread(target=work, daemon=True).start()
    while (item := q.get()) is not None:
        yield item


def to_gpu(faces, device, chunk=2000):
    out = torch.empty(faces.shape, dtype=torch.uint8, device=device)
    for i in range(0, len(faces), chunk):
        out[i:i + chunk] = torch.from_numpy(np.ascontiguousarray(faces[i:i + chunk])).to(device)
    return out


def prep(xb, device, res):
    x = xb.to(device, non_blocking=True).permute(0, 3, 1, 2).float().div_(255)
    if res != x.shape[-1]:
        x = F.interpolate(x, size=(res, res), mode='bilinear', antialias=True, align_corners=False)
    return x


def norm(x):
    return ((x - MEAN.to(x.device)) / STD.to(x.device)).contiguous(memory_format=torch.channels_last)


def augment(x, y, cfg):
    B, dev = x.shape[0], x.device
    y = y.clone()
    if cfg.get('flip', True):  # mirror image <=> negate yaw
        m = torch.rand(B, device=dev) < 0.5
        x = torch.where(m[:, None, None, None], x.flip(3), x)
        y[:, 1] = torch.where(m, -y[:, 1], y[:, 1])
    s, t = cfg.get('scale', 0.0), cfg.get('shift', 0.0)
    if s or t:
        theta = torch.zeros(B, 2, 3, device=dev)
        sc = 1 + (torch.rand(B, device=dev) * 2 - 1) * s
        theta[:, 0, 0], theta[:, 1, 1] = sc, sc
        theta[:, :, 2] = (torch.rand(B, 2, device=dev) * 2 - 1) * t
        x = F.grid_sample(x, F.affine_grid(theta, list(x.shape), align_corners=False), align_corners=False)
    c = cfg.get('color', 0.0)
    if c:
        r = lambda: 1 + (torch.rand(B, 1, 1, 1, device=dev) * 2 - 1) * c
        mean = x.mean((1, 2, 3), keepdim=True)
        x = ((x - mean) * r() + mean) * r()
        gray = x.mean(1, keepdim=True)
        x = (x - gray) * r() + gray
        g = torch.rand(B, device=dev) < cfg.get('gray', 0.1)
        x = torch.where(g[:, None, None, None], gray.expand_as(x), x).clamp(0, 1)
    p = cfg.get('lowres', 0.0)
    if p:  # simulate small video tiles: downscale then upscale
        m = torch.rand(B, device=dev) < p
        if m.any():
            H = x.shape[-1]
            r = int(torch.randint(cfg.get('lowres_min', 24), H, (1,)))
            sub = F.interpolate(x[m], size=(r, r), mode='area')
            if cfg.get('jpeg') and r <= cfg.get('jpeg_max', 112):  # video-call compression on small tiles
                j = (torch.rand(len(sub)) < cfg['jpeg']).to(dev)
                if j.any():
                    sub[j] = jpeg_cpu(sub[j], cfg.get('jpeg_q', (20, 70)))
            x[m] = F.interpolate(sub, size=(H, H), mode='bilinear', align_corners=False)
    return x, y


def jpeg_cpu(x, q_range):
    # JPEG round trip with OpenCV for a small batch of float images in [0, 1] (N, 3, h, w) on any device.
    a = (x.permute(0, 2, 3, 1) * 255).round().clamp(0, 255).to(torch.uint8).cpu().numpy()
    qs = np.random.randint(q_range[0], q_range[1] + 1, len(a))
    out = [cv2.imdecode(cv2.imencode('.jpg', np.ascontiguousarray(im[:, :, ::-1]), [cv2.IMWRITE_JPEG_QUALITY, int(q)])[1],
                        cv2.IMREAD_COLOR)[:, :, ::-1] for im, q in zip(a, qs)]
    return torch.from_numpy(np.ascontiguousarray(np.stack(out))).to(x.device).permute(0, 3, 1, 2).float() / 255


def loss_fn(out, y, kind):
    if kind == 'angular':
        cos = F.cosine_similarity(to_vec(out), to_vec(y)).clamp(-1 + 1e-6, 1 - 1e-6)
        return torch.arccos(cos).mean()
    return F.l1_loss(out, y)


def degrade_img(im, size=224, jpeg=None, **_):
    h = im.shape[0]
    if size < h:
        im = cv2.resize(im, (size, size), interpolation=cv2.INTER_AREA)
    if jpeg:
        enc = cv2.imencode('.jpg', np.ascontiguousarray(im[:, :, ::-1]), [cv2.IMWRITE_JPEG_QUALITY, int(jpeg)])[1]
        im = cv2.imdecode(enc, cv2.IMREAD_COLOR)[:, :, ::-1]
    if im.shape[0] != h:
        im = cv2.resize(im, (h, h), interpolation=cv2.INTER_LINEAR)
    return np.ascontiguousarray(im)


@torch.no_grad()
def predict(model, faces, gaze, idx, device, res=224, tta=True, degrade=None, bs=256):
    model.eval()
    idx = np.asarray(idx)
    out, pos = np.zeros((len(idx), 2), np.float32), {v: i for i, v in enumerate(idx)}
    for xb, _, b in batches(faces, gaze, idx, bs, False):
        if degrade:
            xb = torch.from_numpy(np.stack([degrade_img(im, **degrade) for im in xb.cpu().numpy()]))
        x = norm(prep(xb, device, res))
        with torch.autocast('cuda', dtype=torch.float16):
            p = model(x).float()
            if tta:
                pf = model(x.flip(3)).float()
                p = (p + pf * torch.tensor([1.0, -1.0], device=device)) / 2
        out[[pos[i] for i in b]] = p.cpu().numpy()
    return out


def train_fold(cfg, faces, gaze, train_idx, test_idx, device, log=print, seed=0):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = build(cfg).to(device).to(memory_format=torch.channels_last)
    bs, epochs, res = cfg.get('bs', 64), cfg['epochs'], cfg.get('res', 224)
    steps = epochs * (len(train_idx) // bs)
    head = {id(p) for p in model.get_classifier().parameters()}
    groups = [{'params': [p for p in model.parameters() if id(p) not in head], 'lr': cfg['lr']},
              {'params': [p for p in model.parameters() if id(p) in head], 'lr': cfg['lr'] * cfg.get('head_mult', 10)}]
    opt = torch.optim.AdamW(groups, weight_decay=cfg.get('wd', 0.05))
    warm = max(1, int(cfg.get('warmup', 0.05) * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / steps))))
    scaler = torch.amp.GradScaler()
    ema = timm.utils.ModelEmaV3(model, decay=cfg['ema']) if cfg.get('ema') else None
    probe = np.sort(rng.choice(test_idx, min(1500, len(test_idx)), replace=False)) if len(test_idx) else None
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        tot, n = torch.zeros((), device=device), 0
        for xb, yb, _ in batches(faces, gaze, train_idx, bs, True, rng):
            x, y = augment(prep(xb, device, res), yb.to(device, non_blocking=True), cfg)
            with torch.autocast('cuda', dtype=torch.float16):
                out = model(norm(x))
            loss = loss_fn(out.float(), y, cfg.get('loss', 'l1'))
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            if ema is not None:
                ema.update(model)
            tot += loss.detach() * len(y)
            n += len(y)
        final = ema.module if ema is not None else model
        msg = f'ep {ep + 1}/{epochs} loss {float(tot) / n:.4f} {time.time() - t0:.0f}s'
        if cfg.get('probe', True) and probe is not None:
            msg += f' probe {ang_err(predict(final, faces, gaze, probe, device, res, tta=False), gaze[probe]).mean():.2f}'
        log(msg)
    return ema.module if ema is not None else model

In [ ]:
%%writefile /kaggle/working/run.py
# Train and evaluate one cross-validation fold: python run.py RUN_DIR FOLD_INDEX (GPU via CUDA_VISIBLE_DEVICES).
import json, os, sys, time
import numpy as np, torch
sys.path.insert(0, '/kaggle/working')
from gazelib import ang_err, load, predict, to_gpu, train_fold

run_dir, k = sys.argv[1], int(sys.argv[2])
cfg = json.load(open(f'{run_dir}/cfg.json'))
torch.backends.cudnn.benchmark = True
faces, gaze, person = load()
device = torch.device('cuda', 0)
if cfg.get('gpu_data'):
    faces = to_gpu(faces, device)
test = cfg['folds'][k]
test_idx, train_idx = np.where(np.isin(person, test))[0], np.where(~np.isin(person, test))[0]
with open(f'{run_dir}/log.txt', 'a', buffering=1) as log:
    t = time.time()
    model = train_fold(cfg, faces, gaze, train_idx, test_idx, device, log=lambda s: print(f'fold{k} {s}', file=log),
                       seed=cfg.get('seed', 0))
    if len(test_idx) == 0:  # deployment model: trained on everyone, nothing held out
        torch.save(model.state_dict(), f'{run_dir}/model{k}.pt')
        print(f'fold{k} DONE (no test set) {time.time() - t:.0f}s', file=log)
        sys.exit(0)
    res, res_dict = cfg.get('res', 224), {}
    res_dict['clean'] = predict(model, faces, gaze, test_idx, device, res)
    for d in cfg.get('degrade', []):
        res_dict[d['tag']] = predict(model, faces, gaze, test_idx, device, res, degrade=d)
    np.savez(f'{run_dir}/fold{k}.npz', idx=test_idx, **res_dict)
    if cfg.get('save_model'):
        torch.save(model.state_dict(), f'{run_dir}/model{k}.pt')
    print(f'fold{k} DONE test={test} err={ang_err(res_dict["clean"], gaze[test_idx]).mean():.2f} '
          f'{time.time() - t:.0f}s', file=log)

## 3. Sanity checks
Label ranges, faces with their gaze arrows, and the ETH-XGaze model zero-shot. The zero-shot error only comes out low (about 7 deg) if the normalisation and the pitch/yaw conventions are right; flipping a sign roughly triples it.

In [ ]:
# sanity check: label ranges, normalised faces with gaze arrows, ETH-XGaze model zero-shot (conventions test)
import matplotlib.pyplot as plt, importlib, gazelib
importlib.reload(gazelib)
from gazelib import *
faces, gaze, person = load()
d = np.degrees(gaze)
for j, n in enumerate(['pitch', 'yaw']):
    print(f'{n}: mean {d[:, j].mean():.1f} sd {d[:, j].std():.1f} range {d[:, j].min():.0f}..{d[:, j].max():.0f}', end=' | ')
print('N', len(d), 'people', np.bincount(person).tolist())
fig, ax = plt.subplots(2, 8, figsize=(16, 4.4))
for a, i in zip(ax.flat, np.random.default_rng(1).choice(len(d), 16, replace=False)):
    a.imshow(faces[i]); p, y = gaze[i]
    a.arrow(112, 112, -100 * np.cos(p) * np.sin(y), -100 * np.sin(p), color='lime', width=3)
    a.set_title(f'p{person[i]:02d} {d[i, 0]:.0f},{d[i, 1]:.0f}', fontsize=9); a.axis('off')
plt.tight_layout(); plt.show()
dev = torch.device('cuda', 0)
eth = build({'backbone': 'eth_resnet18'}).to(dev).to(memory_format=torch.channels_last)
sub = np.sort(np.random.default_rng(0).choice(len(d), 3000, replace=False))
pr = predict(eth, faces, gaze, sub, dev)
for tag, q in [('as is', pr), ('yaw sign flipped', pr * [1, -1]), ('pitch sign flipped', pr * [-1, 1]), ('swapped', np.ascontiguousarray(pr[:, ::-1]))]:
    print(f'ETH-XGaze ResNet-18 zero-shot, {tag}: {ang_err(q, gaze[sub]).mean():.2f} deg')
del eth; torch.cuda.empty_cache()

In [ ]:
# The official ETH-XGaze ResNet-50 (Zhang et al. 2020, linked from github.com/xucong-zhang/ETH-XGaze).
# weights_only=True: the safe loader accepts only tensors and plain containers, so no pickled code can run.
# Converted once to timm key names + safetensors; gazelib.build('eth_resnet50') loads that file.
import subprocess, timm, torch
from safetensors.torch import save_file
if not os.path.exists('/tmp/eth_xgaze_r50.safetensors'):
    subprocess.run('pip install -q gdown; gdown -q 1Ma6zJrECNTjo_mToZ5GKk7EF-0FS4nEC -O /tmp/eth_xgaze_r50.pth.tar', shell=True, check=True)
    sd = torch.load('/tmp/eth_xgaze_r50.pth.tar', map_location='cpu', weights_only=True)['model_state']
    sd = {k.replace('gaze_network.', '').replace('gaze_fc.0.', 'fc.'): v.contiguous() for k, v in sd.items()
          if not k.startswith('gaze_network.fc.')}  # drop the unused ImageNet head
    print(timm.create_model('resnet50', pretrained=False, num_classes=2).load_state_dict(sd, strict=True))
    save_file(sd, '/tmp/eth_xgaze_r50.safetensors')
m50 = gazelib.build({'backbone': 'eth_resnet50'}).to(dev).to(memory_format=torch.channels_last)
print(f"ETH-XGaze ResNet-50 zero-shot: {ang_err(gazelib.predict(m50, faces, gaze, sub, dev), gaze[sub]).mean():.2f} deg")
del m50; torch.cuda.empty_cache()

## 4. Job queue
Each (run, fold) is a separate process on a free GPU. The face array is copied into GPU memory per process (`gpu_data`), which keeps both T4s at about 99% utilisation.

In [ ]:
# job queue. Each (config, fold) runs as its own process on a free GPU; results land in /kaggle/working/runs.
import json, os, queue, subprocess, sys, threading, glob
import numpy as np, torch
sys.path.insert(0, '/kaggle/working')
from gazelib import ang_err
RUNS = '/kaggle/working/runs'
FOLDS = {'dev3': [[0, 1, 2, 3, 4], [5, 6, 7, 8, 9], [10, 11, 12, 13, 14]], 'lopo': [[i] for i in range(15)]}
META = np.load('/tmp/mpii_cache/meta.npz')
GAZE, PERSON = META['gaze'].astype(np.float32), META['person']
try:
    JOBS
except NameError:
    JOBS = queue.Queue()
    def _worker(g):
        while True:
            run_dir, k = JOBS.get()
            with open(f'{run_dir}/out{k}.txt', 'w') as f:
                subprocess.run([sys.executable, '/kaggle/working/run.py', run_dir, str(k)], stdout=f,
                               stderr=subprocess.STDOUT, env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(g)))
            JOBS.task_done()
    for g in range(torch.cuda.device_count()):
        threading.Thread(target=_worker, args=(g,), daemon=True).start()

def submit(name, cfg, folds='dev3'):
    run_dir = f'{RUNS}/{name}'
    os.makedirs(run_dir, exist_ok=True)
    json.dump(dict(cfg, folds=FOLDS[folds] if isinstance(folds, str) else folds), open(f'{run_dir}/cfg.json', 'w'))
    for k in range(len(json.load(open(f'{run_dir}/cfg.json'))['folds'])):
        if not os.path.exists(f'{run_dir}/fold{k}.npz'):
            JOBS.put((run_dir, k))

def per_person(name, tag='clean'):
    out = {}
    for f in sorted(glob.glob(f'{RUNS}/{name}/fold*.npz')):
        d = np.load(f)
        if tag not in d: continue
        e, p = ang_err(d[tag], GAZE[d['idx']]), PERSON[d['idx']]
        out.update({int(q): float(e[p == q].mean()) for q in np.unique(p)})
    return out

def status(tail=1):
    for r in sorted(glob.glob(f'{RUNS}/*/')):
        name, cfg = os.path.basename(r[:-1]), json.load(open(f'{r}cfg.json'))
        pp, nf = per_person(name), len(cfg['folds'])
        done = len(glob.glob(f'{r}fold*.npz'))
        mean = f'{np.mean(list(pp.values())):.2f}' if len(pp) == 15 else '-'
        lines = open(f'{r}log.txt').read().strip().splitlines()[-tail:] if os.path.exists(f'{r}log.txt') else []
        print(f'{name:28s} {done}/{nf} folds  mean={mean:5s}  | ' + ' || '.join(lines))
    print('queued:', JOBS.qsize())

## 5. Development experiments (3-fold, split by person)

In [ ]:
# Development experiments (3-fold person CV). Each submit() queues 3 background jobs; finished folds are skipped,
# so re-running this cell only fills in what's missing.
# Round 1: which pretrained starting point transfers best (8 epochs, L1 on pitch/yaw, flip/shift/scale/colour augmentation)
BASE = dict(epochs=8, bs=64, lr=1e-4, head_mult=10, wd=0.05, flip=True, shift=0.05, scale=0.05, color=0.3, gray=0.1,
            loss='l1', gpu_data=True)
submit('r18_imnet', dict(BASE, backbone='resnet18'))                     # ImageNet ResNet-18
submit('eth_r18', dict(BASE, backbone='eth_resnet18', head_mult=1))      # ETH-XGaze ResNet-18 (gaze head kept)
submit('eth_r50', dict(BASE, backbone='eth_resnet50', head_mult=1))      # ETH-XGaze ResNet-50
# Round 2 (ETH-XGaze ResNet-18): 15 epochs + weight EMA, without / with low-resolution + JPEG augmentation.
# Both are also tested on smaller faces (s = face width in px) and video-call-like JPEG compression (q30).
DEG = [dict(tag=f's{s}', size=s) for s in (160, 112, 80, 64, 48, 32)] + [dict(tag=f's{s}_q30', size=s, jpeg=30) for s in (112, 64)]
ETH18 = dict(BASE, backbone='eth_resnet18', head_mult=1, epochs=15, ema=0.999, degrade=DEG)
submit('eth_r18_e15_ema', ETH18)
submit('eth_r18_e15_ema_lowres_jpeg', dict(ETH18, lowres=0.5, lowres_min=24, jpeg=0.5, jpeg_q=(20, 70)))

## 6. Baseline: the current OpenMeet gaze pipeline
MediaPipe Face Landmarker iris offsets (as in `readgaze/landmarks.py`), mapped to gaze angles by a linear fit.

In [ ]:
%%writefile /kaggle/working/mp_baseline.py
# Baseline = the current OpenMeet gaze pipeline (MediaPipe iris offsets, readgaze/landmarks.py) on the same faces,
# 500 random faces per person, at full size and at smaller face sizes.
import sys
sys.path.insert(0, '/tmp/mp')
import cv2, numpy as np, mediapipe as mp
from mediapipe.tasks import python as mpt
from mediapipe.tasks.python import vision

faces = np.load('/tmp/mpii_cache/faces.npy', mmap_mode='r')
person = np.load('/tmp/mpii_cache/meta.npz')['person']
rng = np.random.default_rng(0)
idx = np.sort(np.concatenate([rng.choice(np.where(person == p)[0], 500, replace=False) for p in range(15)]))
lm = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
    base_options=mpt.BaseOptions(model_asset_path='/tmp/face_landmarker.task'),
    running_mode=vision.RunningMode.IMAGE, num_faces=1))
EYES = [(33, 133, 159, 145, 468), (362, 263, 386, 374, 473)]  # corners, lids, iris centre (both eyes)
res = {}
for s in [224, 96, 64, 48, 32]:
    out = np.full((len(idx), 3), np.nan, np.float32)  # horizontal offset, vertical offset, openness
    for j, i in enumerate(idx):
        im = np.ascontiguousarray(faces[i])
        if s < 224:
            im = cv2.resize(cv2.resize(im, (s, s), interpolation=cv2.INTER_AREA), (224, 224))
        pad = cv2.copyMakeBorder(im, 64, 64, 64, 64, cv2.BORDER_CONSTANT)
        r = lm.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=pad))
        if not r.face_landmarks:
            continue
        P = np.array([(q.x * pad.shape[1], q.y * pad.shape[0]) for q in r.face_landmarks[0]])
        h, v, o = [], [], []
        for lc, rc, up, lo, ic in EYES:
            axis = P[rc] - P[lc]
            w = np.hypot(*axis)
            a = axis / w
            off = P[ic] - (P[lc] + P[rc]) / 2
            h.append(off @ a / w); v.append(off @ np.array([-a[1], a[0]]) / w); o.append(np.hypot(*(P[lo] - P[up])) / w)
        out[j] = np.mean(h), np.mean(v), np.mean(o)
    res[f's{s}'] = out
    print(s, 'px: face found', np.isfinite(out[:, 0]).mean().round(3), flush=True)
np.savez('/kaggle/working/mp_baseline.npz', idx=idx, **res)
print('done')

In [ ]:
# Install MediaPipe into its own folder (--no-deps, so the training environment is untouched) and run the baseline
# on the CPU while the GPUs train.
import sys, subprocess
subprocess.run('pip install -q --no-deps --target /tmp/mp mediapipe==0.10.35; '
               'wget -q -O /tmp/face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task',
               shell=True)
MP = subprocess.Popen([sys.executable, '/kaggle/working/mp_baseline.py'], stdout=open('/kaggle/working/mp_log.txt', 'w'),
                      stderr=subprocess.STDOUT)

## 7. Final evaluation and deployment model

In [ ]:
# Final configuration: ETH-XGaze ResNet-18, 15-epoch-equivalent training with EMA, plus low-resolution + JPEG
# augmentation (chosen for the use case: OpenMeet sees small, compressed meeting tiles).
# 11 epochs on 14 people is about the same number of steps as 15 epochs on 10 people in development.
FINAL = dict(ETH18, lowres=0.5, lowres_min=24, jpeg=0.5, jpeg_q=(20, 70), epochs=11, probe=False)
submit('final_lopo', FINAL, 'lopo')  # 15 folds, one held-out person each: the reported number
submit('final_all', FINAL, [[]])     # trained on all 15 people: the model that ships (exported to ONNX below)

In [ ]:
# Wait for every queued training job and the MediaPipe baseline, then show where each run stands
JOBS.join()
MP.wait()
status(1)

## 8. Results

In [ ]:
# Results: every run, mean of per-person angular errors (degrees), clean and degraded test faces
import pandas as pd
def table():
    rows = []
    for r in sorted(glob.glob(f'{RUNS}/*/')):
        name, cfg = os.path.basename(r[:-1]), json.load(open(f'{r}cfg.json'))
        row = {'run': name, 'protocol': 'lopo' if len(cfg['folds']) == 15 else 'dev3'}
        for t in ['clean'] + [d['tag'] for d in cfg.get('degrade', [])]:
            pp = per_person(name, t)
            if len(pp) == 15: row[t] = round(float(np.mean(list(pp.values()))), 2)
        if 'clean' in row: rows.append(row)
    return pd.DataFrame(rows).set_index('run')
RESULTS = table(); print(RESULTS.to_string())

In [ ]:
# MediaPipe iris-offset baseline: map (horizontal, vertical) offsets to gaze with a linear fit.
# "uncalibrated" = fit on the other 14 people (what OpenMeet would do); "calibrated" = fit on half of the
# same person's faces (an upper bound that needs a per-person calibration step). r = corr(offset_x, true yaw).
from sklearn.linear_model import LinearRegression
MPB = np.load('/kaggle/working/mp_baseline.npz'); MIDX = MPB['idx']

def mp_eval(feat):
    ok, P, G = np.isfinite(feat[:, 0]), PERSON[MIDX], GAZE[MIDX]
    unc, cal, r = [], [], []
    for p in range(15):
        te, tr = (P == p) & ok, (P != p) & ok
        X = feat[:, :2]
        unc.append(ang_err(LinearRegression().fit(X[tr], G[tr]).predict(X[te]), G[te]).mean())
        ii = np.where(te)[0]
        cal.append(ang_err(LinearRegression().fit(X[ii[::2]], G[ii[::2]]).predict(X[ii[1::2]]), G[ii[1::2]]).mean())
        r.append(abs(np.corrcoef(X[te, 0], G[te, 1])[0, 1]))
    return dict(found=ok.mean(), uncalibrated=np.mean(unc), calibrated=np.mean(cal), r_yaw=np.mean(r))

for k in sorted(MPB.files, key=lambda s: -int(s[1:]) if s != 'idx' else 0):
    if k != 'idx':
        print(f'MediaPipe {k[1:]:>3s}px', {a: round(float(b), 3) for a, b in mp_eval(MPB[k]).items()})

In [ ]:
# Robustness: error vs face size, learned model (final LOPO run) vs the MediaPipe baseline, on the same 7,500 faces.
# cnn_r_yaw / mp_r_yaw = within-person correlation with true yaw: what the reading (sawtooth) detector needs.
def full_preds(name):
    out = {}
    for f in sorted(glob.glob(f'{RUNS}/{name}/fold*.npz')):
        d = np.load(f)
        for t in d.files:
            if t != 'idx':
                out.setdefault(t, np.full((len(GAZE), 2), np.nan, np.float32))[d['idx']] = d[t]
    return out

def cnn_stats(pred, idx):
    P, G, Q = PERSON[idx], GAZE[idx], pred[idx]
    e = ang_err(Q, G)
    return (np.mean([e[P == p].mean() for p in range(15)]),
            np.mean([np.corrcoef(Q[P == p, 1], G[P == p, 1])[0, 1] for p in range(15)]))

FINAL_RUN = 'final_lopo'
PRED = full_preds(FINAL_RUN)
rows = []
for s in [224, 160, 112, 96, 80, 64, 48, 32]:
    tag = 'clean' if s == 224 else f's{s}'
    row = {'face_px': s}
    if tag in PRED:
        row['cnn_err'], row['cnn_r_yaw'] = cnn_stats(PRED[tag], MIDX)
    if f's{s}' in MPB.files:
        mp = mp_eval(MPB[f's{s}'])
        row.update(mp_found=mp['found'], mp_err_uncal=mp['uncalibrated'], mp_err_cal=mp['calibrated'], mp_r_yaw=mp['r_yaw'])
    rows.append(row)
ROB = pd.DataFrame(rows).round(3)
print(ROB.to_string(index=False))
for t in ['s112_q30', 's64_q30']:
    if t in PRED: print(t, 'cnn_err %.2f  r_yaw %.3f' % cnn_stats(PRED[t], MIDX))

os.makedirs('/kaggle/working/results', exist_ok=True)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
c, m = ROB.dropna(subset=['cnn_err']), ROB.dropna(subset=['mp_err_uncal'])
ax[0].plot(c.face_px, c.cnn_err, 'o-', label='transfer-learned CNN (LOPO)')
ax[0].plot(m.face_px, m.mp_err_uncal, 's--', label='MediaPipe iris offsets, uncalibrated')
ax[0].plot(m.face_px, m.mp_err_cal, 's:', label='MediaPipe iris offsets, per-person calibrated')
ax[0].set(xlabel='face width in pixels', ylabel='mean angular error (deg)', xscale='log', title='Gaze error vs face size'); ax[0].legend(fontsize=8)
ax[1].plot(c.face_px, c.cnn_r_yaw, 'o-', label='CNN'); ax[1].plot(m.face_px, m.mp_r_yaw, 's--', label='MediaPipe')
ax[1].set(xlabel='face width in pixels', ylabel='within-person corr. with true yaw', xscale='log', ylim=(0, 1), title='Horizontal gaze signal'); ax[1].legend(fontsize=8)
for a in ax: a.set_xticks([32, 48, 64, 112, 224]); a.set_xticklabels([32, 48, 64, 112, 224]); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig('/kaggle/working/results/robustness.png', dpi=130); plt.show()

In [ ]:
# Deployment model -> ONNX. Preprocessing is inside the graph: input = normalised face crop, uint8 RGB (N, 224, 224, 3);
# output = gaze (pitch, yaw) in radians. Checked against PyTorch, then timed on this machine's CPU (batch 1).
import time
subprocess.run('pip install -q onnx onnxruntime 2>&1 | tail -1', shell=True)
import onnxruntime as ort

class Deploy(torch.nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m
        self.register_buffer('mean', gazelib.MEAN * 255)
        self.register_buffer('std', gazelib.STD * 255)

    def forward(self, face):
        return self.m((face.permute(0, 3, 1, 2).float() - self.mean) / self.std)

DEPLOY_RUN = 'final_all'
cfg = json.load(open(f'{RUNS}/{DEPLOY_RUN}/cfg.json'))
arch = {'eth_resnet18': 'resnet18', 'eth_resnet50': 'resnet50'}.get(cfg['backbone'], cfg['backbone'])
net = timm.create_model(arch, pretrained=False, num_classes=2)
net.load_state_dict(torch.load(f'{RUNS}/{DEPLOY_RUN}/model0.pt', map_location='cpu'))  # our own weights file
dep = Deploy(net).eval()
ONNX = '/kaggle/working/openmeet_gaze_resnet18.onnx'
x = torch.from_numpy(np.ascontiguousarray(faces[:16]))
try:  # classic TorchScript exporter
    torch.onnx.export(dep, (x[:1],), ONNX, input_names=['face'], output_names=['pitch_yaw'],
                      dynamic_axes={'face': {0: 'n'}, 'pitch_yaw': {0: 'n'}}, opset_version=17, dynamo=False)
except Exception as e:  # newer torch: the dynamo exporter (needs onnxscript)
    print('classic exporter failed:', type(e).__name__, '- using dynamo')
    subprocess.run('pip install -q onnxscript 2>&1 | tail -1', shell=True)
    torch.onnx.export(dep, (x[:1],), ONNX, input_names=['face'], output_names=['pitch_yaw'],
                      dynamic_shapes=({0: torch.export.Dim('n', max=256)},), dynamo=True)
with torch.no_grad():
    ref = dep(x).numpy()
ONNX_INFO = {'file': os.path.basename(ONNX), 'mb': round(os.path.getsize(ONNX) / 1e6, 1)}
for threads in (1, 2, 4):
    so = ort.SessionOptions(); so.intra_op_num_threads = threads
    sess = ort.InferenceSession(ONNX, so, providers=['CPUExecutionProvider'])
    ONNX_INFO['max_abs_diff_rad'] = float(np.abs(sess.run(None, {'face': x.numpy()})[0] - ref).max())
    one = x[:1].numpy()
    for _ in range(10): sess.run(None, {'face': one})
    t = time.perf_counter()
    for _ in range(100): sess.run(None, {'face': one})
    ONNX_INFO[f'cpu_ms_{threads}thread'] = round((time.perf_counter() - t) * 10, 2)
print(ONNX_INFO)

In [ ]:
# Save every reported number in one JSON (committed to the repo under training/gaze-detection/results/)
OUT = '/kaggle/working/results'
os.makedirs(OUT, exist_ok=True)
pp = per_person(FINAL_RUN)
SUMMARY = {
    'dataset': 'MPIIFaceGaze: 37,667 faces of 15 people, normalised to 224x224 (focal 960, 600 mm)',
    'metric': 'angular gaze error in degrees; mean over people of each person\'s mean',
    'dev_3fold': RESULTS.reset_index().to_dict('records'),
    'final_lopo': {'run': FINAL_RUN, 'mean': float(np.mean(list(pp.values()))), 'sd_over_people': float(np.std(list(pp.values()))),
                   'per_person': pp, 'config': {k: v for k, v in json.load(open(f'{RUNS}/{FINAL_RUN}/cfg.json')).items() if k not in ('folds', 'degrade')}},
    'robustness_vs_mediapipe': ROB.to_dict('records'),
    'compressed': {t: dict(zip(['cnn_err', 'cnn_r_yaw'], map(float, cnn_stats(PRED[t], MIDX)))) for t in ('s112_q30', 's64_q30') if t in PRED},
    'mediapipe_baseline': {k: {a: float(b) for a, b in mp_eval(MPB[k]).items()} for k in MPB.files if k != 'idx'},
    'onnx': ONNX_INFO,
}
json.dump(SUMMARY, open(f'{OUT}/gaze_model_results.json', 'w'), indent=1, default=float)
print(json.dumps(SUMMARY, default=float))